# 05 Model Evaluation

Goal: evaluate the selected model with threshold optimization, calibration, ROC, PR, and confusion-matrix metrics.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_preprocessing import TARGET_COLUMN
from evaluation import binary_classification_metrics, plot_calibration_curve, plot_roc_and_pr_curves, threshold_optimization_table
from model_training import split_features_target

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "model_ready_data.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "models" / "final_model.joblib"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
split = split_features_target(df, target_column=TARGET_COLUMN, test_size=0.2, random_state=42)
final_model = joblib.load(MODEL_PATH)
y_proba = final_model.predict_proba(split.X_test)[:, 1]
binary_classification_metrics(split.y_test, y_proba, threshold=0.5)

In [ ]:
threshold_table = threshold_optimization_table(split.y_test, y_proba, false_negative_cost=5, false_positive_cost=1)
threshold_table.to_csv(TABLES_DIR / "threshold_comparison.csv", index=False)
threshold_table.head(10)

In [ ]:
plot_roc_and_pr_curves({"final_model": y_proba}, split.y_test, FIGURES_DIR)
plot_calibration_curve(final_model, split.X_test, split.y_test, FIGURES_DIR / "calibration_curve.png", model_name="Final model")
